# Chapter 3: Agent Harnesses and Execution Loops
## Building and Hardening an Agent Loop - Code Examples

This notebook contains the complete, runnable code from Chapter 3. It builds an agent harness from the loop outward on the OpenAI Responses API:
- A minimal harness: a tool definition, a dispatcher, and the loop that ties sampling, parsing, dispatch, and observation together
- Termination and budgets: classifying how a model call ended, and step, token, cost, and wall-clock limits the harness enforces itself
- Oscillation detection and the premature-exit problem, with verification and reinjection
- Context across a long run: clearing old tool results, progress files, and checkpoints that survive a crash
- Tools written for a model: bounded reads and error messages that say what to do next
- Control and diagnosis: permission modes, an authorization gate, and OpenTelemetry-style trace records
- The hardened loop assembled end to end, fixing a real bug in a sample repository

### Setup

The dependencies for every chapter are declared in `pyproject.toml` at the repository root. From the root, run:

```bash
uv sync --all-groups
```

Then start Jupyter with `uv run jupyter lab` and select the **Agentic AI Handbook (Python 3.13)** kernel.

This notebook calls the OpenAI API. Copy `.env.example` to `.env` at the repository root and add your `OPENAI_API_KEY` before running the cells.

The printed listings in the chapter name `gpt-6-astra`. This notebook uses `gpt-5.6-luna`, and any model that supports function calling on the Responses API will run the same code. Every cell writes only inside a temporary workspace created in the setup cell, so nothing in this repository is modified.

In [ ]:
# Import required libraries
import os
import sys
import json
import time
import tempfile
import subprocess
from pathlib import Path
from types import SimpleNamespace
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"

# The verifier in Part 2 shells out to pytest; plain output reads better in a notebook.
os.environ["NO_COLOR"] = "1"

# Every file operation in this notebook happens inside a throwaway workspace
# that holds a tiny sample repository with one deliberate bug. The harness
# will read it, edit it, run its tests, and keep its notes and checkpoints here.
WORKSPACE = Path(tempfile.mkdtemp(prefix="ch3-harness-")).resolve()
os.chdir(WORKSPACE)

(WORKSPACE / "calc").mkdir()
(WORKSPACE / "calc" / "__init__.py").write_text("")
(WORKSPACE / "calc" / "stats.py").write_text(
    '''"""Small statistics helpers for the sample repository."""


def mean(values: list[float]) -> float:
    """Return the arithmetic mean of a non-empty list."""
    return sum(values) / len(values)


def median(values: list[float]) -> float:
    """Return the median of a non-empty list."""
    ordered = sorted(values)
    n = len(ordered)
    mid = n // 2
    if n % 2 == 1:
        return ordered[mid]
    return (ordered[mid] + ordered[mid + 1]) / 2
'''
)
(WORKSPACE / "tests").mkdir()
(WORKSPACE / "tests" / "test_stats.py").write_text(
    '''from calc.stats import mean, median


def test_mean():
    assert mean([1, 2, 3, 4]) == 2.5


def test_median_odd():
    assert median([3, 1, 2]) == 2


def test_median_even():
    assert median([1, 2, 3, 4]) == 2.5
'''
)
# An empty conftest.py at the root makes the `calc` package importable from tests/.
(WORKSPACE / "conftest.py").write_text("")

print("Environment setup complete")
print(f"Workspace: {WORKSPACE}")

## Part 1: A Minimal Harness

A harness starts with a tool the model can call and a dispatcher that runs it. The `read_file` tool below is declared in the Responses API function-tool shape with `strict` set to true, so the API enforces the JSON Schema on the model's arguments. The tool returns a window of at most 100 lines rather than the whole file, a choice the chapter justifies with the SWE-agent ablations in Part 3.

In [ ]:
import json
from pathlib import Path

TOOLS = [
    {
        "type": "function",
        "name": "read_file",
        "description": "Return up to 100 lines of a text file, starting at line `start`.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "start": {"type": "integer", "minimum": 1},
            },
            "required": ["path", "start"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

def dispatch(name: str, arguments: str) -> str:
    args = json.loads(arguments)
    if name == "read_file":
        lines = Path(args["path"]).read_text().splitlines()
        window = lines[args["start"] - 1 : args["start"] + 99]
        return json.dumps({"lines": window, "total_lines": len(lines)})
    return json.dumps({"error": f"unknown tool {name}"})


# --- Run it ---

print("=== dispatch() on the sample file ===")
result = json.loads(dispatch("read_file", json.dumps({"path": "calc/stats.py", "start": 1})))
print(f"total_lines: {result['total_lines']}")
for number, line in enumerate(result["lines"], 1):
    print(f"{number:3d}  {line}")

### The core loop

The loop runs the same four steps on every iteration: sample the model, parse its tool calls, dispatch them, and format the observations. With `store=False` the API keeps nothing, so the harness resends the full history on every call, including the reasoning items, which `include=["reasoning.encrypted_content"]` returns in a form that can be sent back. Each `function_call` item carries a `name`, `arguments` as a JSON string, and a `call_id` that the matching `function_call_output` must echo.

In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5.6-luna"
SYSTEM = "You are a coding agent. Gather facts with tools before you answer."

def run(task: str, max_steps: int = 20) -> str:
    history = [{"role": "user", "content": task}]
    for step in range(max_steps):
        response = client.responses.create(
            model=MODEL, instructions=SYSTEM, tools=TOOLS,
            input=history, store=False,
            include=["reasoning.encrypted_content"],
        )
        history += response.output
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text
        for call in calls:
            history.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": dispatch(call.name, call.arguments),
            })
    raise RuntimeError(f"no answer after {max_steps} steps")


# --- Run it ---

print("=== run() on a small task ===")
answer = run("Read calc/stats.py and list the functions it defines, with one short "
             "sentence on what each one returns.")
print(answer)

## Part 2: Reading the Termination Signal and Enforcing Budgets

A response with no tool calls is not always a finished task. It might mean the model hit its output limit, refused, or failed. The `classify()` function maps one response to one of five outcomes using the `status` field, `incomplete_details.reason`, and any `refusal` content part. The test below produces a `tool_calls` outcome with a normal request and a `truncated` outcome by capping `max_output_tokens`.

In [ ]:
def classify(response) -> str:
    if response.status == "failed":
        return "failed"
    if response.status == "incomplete":
        reason = response.incomplete_details.reason
        return "truncated" if reason == "max_output_tokens" else "blocked"
    for item in response.output:
        if item.type == "message":
            if any(part.type == "refusal" for part in item.content):
                return "refused"
    if any(item.type == "function_call" for item in response.output):
        return "tool_calls"
    return "done"


# --- Run it ---

print("=== classify() on two responses ===")
response = client.responses.create(
    model=MODEL, instructions=SYSTEM, tools=TOOLS,
    input=[{"role": "user", "content": "Read calc/stats.py starting at line 1 and summarize it."}],
    store=False, include=["reasoning.encrypted_content"],
)
print(f"normal request      -> {classify(response)}")

capped = client.responses.create(
    model=MODEL, instructions=SYSTEM, tools=TOOLS,
    input=[{"role": "user", "content": "Explain what a harness is in three paragraphs."}],
    store=False, max_output_tokens=32,
)
print(f"max_output_tokens=32 -> {classify(capped)}  (status={capped.status}, "
      f"reason={capped.incomplete_details.reason if capped.incomplete_details else None})")

### Budgets the harness enforces itself

The model cannot count its own steps, see the clock, or know the price list, so budgets live in the harness. The `Budget` dataclass tracks steps, tokens, cost, and wall clock in one place. The loop calls `charge()` after every response and `exceeded()` before the next call. The prices below are placeholders; take yours from the provider's price list.

In [ ]:
import time
from dataclasses import dataclass, field, asdict

@dataclass
class Budget:
    max_steps: int = 30
    max_tokens: int = 400_000
    max_cost_usd: float = 2.00
    max_seconds: float = 600.0
    price_in: float = 2.50   # dollars per million input tokens, from your provider's price list
    price_out: float = 10.00  # dollars per million output tokens
    steps: int = 0
    tokens: int = 0
    cost_usd: float = 0.0
    started: float = field(default_factory=time.monotonic)

    def charge(self, usage) -> None:
        self.steps += 1
        self.tokens += usage.total_tokens
        self.cost_usd += (usage.input_tokens * self.price_in
                          + usage.output_tokens * self.price_out) / 1_000_000

    def exceeded(self) -> str | None:
        if self.steps >= self.max_steps:
            return "step budget"
        if self.tokens >= self.max_tokens:
            return "token budget"
        if self.cost_usd >= self.max_cost_usd:
            return "cost budget"
        if time.monotonic() - self.started >= self.max_seconds:
            return "wall-clock budget"
        return None


# --- Run it ---

print("=== Budget after one model call ===")
budget = Budget()
budget.charge(response.usage)
print(f"steps={budget.steps}  tokens={budget.tokens}  cost_usd={budget.cost_usd:.5f}")
print(f"exceeded() -> {budget.exceeded()}")

tiny = Budget(max_steps=1)
tiny.charge(response.usage)
print(f"with max_steps=1, exceeded() -> {tiny.exceeded()}")

### Detecting oscillation and repeated calls

A budget catches a looping agent only after the loop has spent most of the run. The `LoopDetector` canonicalizes each tool call, hashes the name and arguments into a sliding window, and counts edits per file. When a threshold is crossed it returns a message that the harness appends to the history as a user turn, nudging the model rather than hiding the tool. The test feeds it the same call three times using a small stand-in for a `function_call` item.

In [ ]:
import hashlib
from collections import Counter, deque

class LoopDetector:
    def __init__(self, window: int = 8, max_repeats: int = 3, max_edits: int = 6):
        self.recent = deque(maxlen=window)
        self.edits = Counter()
        self.max_repeats, self.max_edits = max_repeats, max_edits

    def observe(self, call) -> str | None:
        args = json.dumps(json.loads(call.arguments), sort_keys=True)
        key = hashlib.sha256(f"{call.name}:{args}".encode()).hexdigest()
        self.recent.append(key)
        if self.recent.count(key) >= self.max_repeats:
            return (f"You have made the same {call.name} call {self.max_repeats} times "
                    "with identical arguments. Change your approach before calling it again.")
        if call.name == "edit_file":
            path = json.loads(call.arguments)["path"]
            self.edits[path] += 1
            if self.edits[path] > self.max_edits:
                return (f"You have edited {path} {self.edits[path]} times. Stop, re-read the "
                        "file, and state what is failing before you edit it again.")
        return None


# --- Run it ---

print("=== LoopDetector on three identical calls ===")
detector = LoopDetector()
same_call = SimpleNamespace(
    name="edit_file",
    arguments=json.dumps({"path": "calc/stats.py", "start": 16, "end": 16, "new_text": "    pass"}),
)
for attempt in range(1, 4):
    nudge = detector.observe(same_call)
    print(f"attempt {attempt}: {nudge or 'no nudge'}")

### The premature-exit problem

A completed response with no tool calls is the model's claim that it is done, not a verdict. The reinjection hook runs a verifier the harness trusts, here `pytest -q` in the workspace, and on failure appends the test output to the history as a new user message so the model can repair the problem. The sample repository still contains its bug, so a claim of completion is rejected below.

In [ ]:
import subprocess

MAX_REINJECTIONS = 2

def verify() -> str | None:
    result = subprocess.run(["pytest", "-q"], capture_output=True, text=True)
    return None if result.returncode == 0 else result.stdout[-2000:]

def on_completion(history: list, response, reinjections: int) -> tuple[str | None, int]:
    failure = verify()
    if failure is None or reinjections >= MAX_REINJECTIONS:
        return response.output_text, reinjections
    history.append({
        "role": "user",
        "content": "You reported completion, but verification failed:\n"
                   + failure + "\nFix the cause, then report again.",
    })
    return None, reinjections + 1


# --- Run it ---

print("=== on_completion() against a failing test suite ===")
claimed_done = SimpleNamespace(output_text="All done. The statistics helpers are correct.")
history = [{"role": "user", "content": "Fix the failing tests."}]
answer, reinjections = on_completion(history, claimed_done, 0)
print(f"accepted answer: {answer}")
print(f"reinjections so far: {reinjections}")
print("\nMessage reinjected into the history:\n")
print(history[-1]["content"])

## Part 3: Context, State, and Tools Across a Long Run

A run of hundreds of turns fills the context window several times over. The mildest form of compaction replaces the `output` field of older tool results with a short marker, keeping the call and output pair in place so the transcript stays valid and the result stays recoverable at the cost of one more call. `estimate_tokens()` uses the same rough characters-divided-by-four estimate as Chapter 2.

In [ ]:
CONTEXT_BUDGET = 400_000
CLEARED = "[output cleared to save context; call the tool again if you still need it]"

def estimate_tokens(history: list) -> int:
    text = "".join(json.dumps(item) if isinstance(item, dict) else item.model_dump_json()
                   for item in history)
    return len(text) // 4

def clear_old_tool_results(history: list, keep_last: int = 6) -> list:
    outputs = [i for i, item in enumerate(history)
               if isinstance(item, dict) and item.get("type") == "function_call_output"]
    for i in outputs[:-keep_last]:
        history[i] = {**history[i], "output": CLEARED}
    return history


# --- Run it ---

print("=== Clearing old tool results ===")
synthetic = [{"role": "user", "content": "Investigate the repository."}]
for i in range(10):
    synthetic.append({"type": "function_call", "call_id": f"call_{i}", "name": "read_file",
                      "arguments": json.dumps({"path": f"file_{i}.py", "start": 1})})
    synthetic.append({"type": "function_call_output", "call_id": f"call_{i}",
                      "output": json.dumps({"lines": ["x = 1"] * 200})})
before = estimate_tokens(synthetic)
cleared = clear_old_tool_results(synthetic, keep_last=3)
after = estimate_tokens(cleared)
print(f"estimated tokens before: {before}   after: {after}")
print(f"oldest output now: {cleared[2]['output']}")
print(f"newest output kept: {cleared[-1]['output'][:40]}...")

### Structured notes and externalized state

Notes written to a file outside the window survive every compaction. The write-and-recite pattern appends one line to `progress.md` whenever a step finishes and attaches the file's contents to the history as a user message before each model call, which pushes the plan back into recent attention.

In [ ]:
PROGRESS = Path("progress.md")

def write_progress(note: str) -> None:
    with PROGRESS.open("a") as f:
        f.write(f"- {note}\n")

def recite(history: list) -> list:
    notes = PROGRESS.read_text() if PROGRESS.exists() else "(nothing recorded yet)"
    reminder = {"role": "user",
                "content": "Progress so far and remaining work, from progress.md:\n" + notes}
    return history + [reminder]


# --- Run it ---

print("=== write_progress() and recite() ===")
write_progress("Read calc/stats.py; the even-length branch of median looks wrong")
write_progress("tests/test_stats.py::test_median_even fails")
reminded = recite([{"role": "user", "content": "Fix the failing tests."}])
print(f"history length: 1 -> {len(reminded)}")
print(reminded[-1]["content"])

### Checkpointing and resumption

A note is written for the model and is lossy by design. A checkpoint is written for the harness and is exact. `save_checkpoint()` converts each history item to a plain dictionary, since SDK items are Pydantic objects, and stores the budget counters and step number beside it. The test dumps a real history, restores it in a fresh set of variables, and sends it back to the API to prove the run resumes. One subtlety: `started` is a monotonic clock reading, so a real harness should reset it on load or store wall-clock time instead.

In [ ]:
def save_checkpoint(path: str, history: list, budget: Budget, step: int) -> None:
    items = [item.model_dump(exclude_none=True) if hasattr(item, "model_dump") else item
             for item in history]
    Path(path).write_text(json.dumps({"history": items, "budget": asdict(budget), "step": step}))

def load_checkpoint(path: str) -> tuple[list, Budget, int]:
    data = json.loads(Path(path).read_text())
    return data["history"], Budget(**data["budget"]), data["step"]


# --- Run it ---

print("=== Checkpoint round trip ===")
# Build a real history: the request from Part 2, the model's tool call, and its result.
history = [{"role": "user", "content": "Read calc/stats.py starting at line 1 and summarize it."}]
history += response.output
for call in (item for item in response.output if item.type == "function_call"):
    history.append({"type": "function_call_output", "call_id": call.call_id,
                    "output": dispatch(call.name, call.arguments)})
save_checkpoint("checkpoint.json", history, budget, step=1)
print(f"checkpoint.json written: {Path('checkpoint.json').stat().st_size} bytes")

restored_history, restored_budget, restored_step = load_checkpoint("checkpoint.json")
restored_budget.started = time.monotonic()   # a fresh process needs a fresh clock
print(f"restored: {len(restored_history)} items, step {restored_step}, "
      f"{restored_budget.tokens} tokens charged so far")

resumed = client.responses.create(
    model=MODEL, instructions=SYSTEM, tools=TOOLS,
    input=restored_history, store=False, include=["reasoning.encrypted_content"],
)
restored_budget.charge(resumed.usage)
print(f"resumed run -> {classify(resumed)}")
print(resumed.output_text)

### Error messages written for a model

A tool's failure message is half of its contract. The `edit_file` tool below validates the candidate file with `compile()` before writing anything. On a syntax error it returns JSON that names the line, tells the model what to re-read, and states that the file was not changed. The test submits a broken edit, then a harmless valid one that replaces the module docstring.

In [ ]:
def edit_file(path: str, start: int, end: int, new_text: str) -> str:
    lines = Path(path).read_text().splitlines()
    candidate = lines[: start - 1] + new_text.splitlines() + lines[end:]
    try:
        compile("\n".join(candidate), path, "exec")
    except SyntaxError as e:
        return json.dumps({
            "ok": False,
            "error": f"SyntaxError at line {e.lineno}: {e.msg}",
            "hint": f"Re-read lines {max(1, e.lineno - 4)} to {e.lineno + 4} with "
                    "read_file and resubmit the whole block. The file was not changed.",
        })
    Path(path).write_text("\n".join(candidate) + "\n")
    return json.dumps({"ok": True, "lines_replaced": end - start + 1})


# --- Run it ---

print("=== edit_file() rejecting a broken edit ===")
print(edit_file("calc/stats.py", 6, 6, "    return (sum(values) / len(values)"))

print("\n=== edit_file() applying a valid edit ===")
print(edit_file("calc/stats.py", 1, 1, '"""Statistics helpers for the Chapter 3 sample repository."""'))
print(Path("calc/stats.py").read_text().splitlines()[0])

## Part 4: Control and Diagnosis

A permission mode maps every tool call to allow, ask, or deny before it runs. The `Mode` enum mirrors the three sandbox levels of Codex CLI, and `authorize()` checks the deny patterns first in every mode, allows reads everywhere, and under `WORKSPACE_WRITE` allows a write only when its target resolves inside the workspace. Matching strings in the serialized arguments is a coarse check for a book example; a production deny list should parse the command.

In [ ]:
from enum import Enum

class Mode(Enum):
    READ_ONLY = "read_only"
    WORKSPACE_WRITE = "workspace_write"
    FULL_ACCESS = "full_access"

READ_TOOLS = {"read_file", "grep", "find"}
DENY_PATTERNS = ("rm -rf", "git push --force", "DROP TABLE")

def authorize(call, mode: Mode, workspace: Path) -> str:
    args = json.loads(call.arguments)
    if any(pattern in json.dumps(args) for pattern in DENY_PATTERNS):
        return "deny"
    if call.name in READ_TOOLS:
        return "allow"
    if mode is Mode.FULL_ACCESS:
        return "allow"
    if mode is Mode.READ_ONLY:
        return "ask"
    target = Path(args.get("path", workspace)).resolve()
    inside = target == workspace.resolve() or workspace.resolve() in target.parents
    return "allow" if inside else "ask"


# --- Run it ---

def fake_call(name: str, **arguments) -> SimpleNamespace:
    return SimpleNamespace(name=name, call_id="call_demo", arguments=json.dumps(arguments))

print("=== authorize() decisions ===")
cases = [
    (Mode.WORKSPACE_WRITE, fake_call("read_file", path="calc/stats.py", start=1)),
    (Mode.WORKSPACE_WRITE, fake_call("edit_file", path="calc/stats.py", start=1, end=1, new_text="")),
    (Mode.WORKSPACE_WRITE, fake_call("edit_file", path="../outside.py", start=1, end=1, new_text="")),
    (Mode.READ_ONLY, fake_call("edit_file", path="calc/stats.py", start=1, end=1, new_text="")),
    (Mode.FULL_ACCESS, fake_call("run_shell", command="rm -rf /")),
]
for mode, call in cases:
    args = json.loads(call.arguments)
    target = args.get("path", args.get("command"))
    print(f"{mode.value:16s} {call.name:10s} {target:20s} -> {authorize(call, mode, WORKSPACE)}")

### Tracing full runs

The tracer is the component most often left out of a first harness and the only observer present at every step. `trace_turn()` builds one record per model call and `trace_tool()` one record per tool call, using attribute names from the OpenTelemetry semantic conventions for generative AI so that other teams' tools can read the traces. The `result_bytes` field is the cheapest early warning of a tool that is flooding the context.

In [ ]:
def trace_turn(response, step: int) -> dict:
    return {
        "gen_ai.operation.name": "invoke_agent",
        "gen_ai.request.model": MODEL,
        "gen_ai.response.id": response.id,
        "gen_ai.usage.input_tokens": response.usage.input_tokens,
        "gen_ai.usage.output_tokens": response.usage.output_tokens,
        "step": step,
        "status": response.status,
    }

def trace_tool(call, started: float, result: str) -> dict:
    return {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": call.name,
        "gen_ai.tool.call.id": call.call_id,
        "duration_ms": round((time.monotonic() - started) * 1000),
        "result_bytes": len(result),
    }


# --- Run it ---

print("=== trace records ===")
print(json.dumps(trace_turn(response, step=1), indent=2))
call = next(item for item in response.output if item.type == "function_call")
started = time.monotonic()
result = dispatch(call.name, call.arguments)
print(json.dumps(trace_tool(call, started, result), indent=2))

## Part 5: The Hardened Loop End to End

This final cell is the only code in the notebook that does not appear as a listing in the chapter. It assembles every component from Parts 1 to 4 into one loop: `classify()` decides what each response means, `Budget` and `LoopDetector` bound the run, `authorize()` gates every call in `WORKSPACE_WRITE` mode, old tool results are cleared past the context budget, the progress file is recited on every turn, a checkpoint is written after every dispatch, `on_completion()` verifies any claim of completion against `pytest`, and every call is traced.

The task is the bug planted in the setup cell. The agent gets `read_file` and `edit_file`, must find the failing test, repair `calc/stats.py`, and report what it changed. The verifier, not the model, decides when the run is finished.

In [ ]:
EDIT_TOOL = {
    "type": "function",
    "name": "edit_file",
    "description": ("Replace lines `start` to `end` (inclusive, 1-based) of a Python file with "
                    "`new_text`. The file is syntax-checked and left unchanged if the edit is invalid."),
    "parameters": {
        "type": "object",
        "properties": {
            "path": {"type": "string"},
            "start": {"type": "integer", "minimum": 1},
            "end": {"type": "integer", "minimum": 1},
            "new_text": {"type": "string"},
        },
        "required": ["path", "start", "end", "new_text"],
        "additionalProperties": False,
    },
    "strict": True,
}
HARDENED_TOOLS = TOOLS + [EDIT_TOOL]


def dispatch_hardened(name: str, arguments: str) -> str:
    """Route edit_file to the validated editor and everything else to dispatch()."""
    if name == "edit_file":
        args = json.loads(arguments)
        return edit_file(args["path"], args["start"], args["end"], args["new_text"])
    return dispatch(name, arguments)


def run_hardened(task: str, mode: Mode = Mode.WORKSPACE_WRITE,
                 checkpoint_path: str = "checkpoint.json") -> tuple[str, list, Budget]:
    """The minimal loop from Part 1 with every hardening step from Parts 2 to 4."""
    history = [{"role": "user", "content": task}]
    budget = Budget(max_steps=15, max_cost_usd=1.00, max_seconds=300.0)
    detector = LoopDetector()
    trace: list[dict] = []
    reinjections = 0
    step = 0

    while True:
        # 1. Budgets are checked by the harness before every call.
        reason = budget.exceeded()
        if reason:
            save_checkpoint(checkpoint_path, history, budget, step)
            return f"stopped: {reason}", trace, budget

        # 2. Compact if the history has outgrown the context budget.
        if estimate_tokens(history) > CONTEXT_BUDGET:
            history = clear_old_tool_results(history)

        # 3. Sample, with the progress file recited into the input.
        response = client.responses.create(
            model=MODEL, instructions=SYSTEM, tools=HARDENED_TOOLS,
            input=recite(history), store=False,
            include=["reasoning.encrypted_content"],
        )
        step += 1
        budget.charge(response.usage)
        trace.append(trace_turn(response, step))
        outcome = classify(response)
        history += response.output

        # 4. Dispatch through the permission gate and the loop detector.
        if outcome == "tool_calls":
            for call in (item for item in response.output if item.type == "function_call"):
                decision = authorize(call, mode, WORKSPACE)
                nudge = detector.observe(call)
                started = time.monotonic()
                if decision == "allow":
                    result = dispatch_hardened(call.name, call.arguments)
                else:
                    result = json.dumps({"ok": False, "error": f"call {decision}ed by the permission layer"})
                trace.append({**trace_tool(call, started, result), "permission": decision})
                history.append({"type": "function_call_output", "call_id": call.call_id, "output": result})
                if nudge:
                    history.append({"role": "user", "content": nudge})
                write_progress(f"step {step}: {call.name} -> {decision}")
            save_checkpoint(checkpoint_path, history, budget, step)
            continue

        # 5. A claim of completion is verified before it is accepted.
        if outcome == "done":
            answer, reinjections = on_completion(history, response, reinjections)
            if answer is not None:
                return answer, trace, budget
            write_progress(f"step {step}: completion claimed, verification failed, reinjected")
            continue

        # 6. Everything else stops the run and hands it to a human.
        save_checkpoint(checkpoint_path, history, budget, step)
        return f"stopped: {outcome}", trace, budget


# --- Run it ---

PROGRESS.unlink(missing_ok=True)
task = ("The test suite in tests/test_stats.py has a failing test. Read tests/test_stats.py "
        "and calc/stats.py, fix the bug in calc/stats.py with edit_file, then report in two "
        "sentences what was wrong and what you changed.")

answer, trace, budget = run_hardened(task)

print("=== FINAL ANSWER ===")
print(answer)

print("\n=== TRACE ===")
for record in trace:
    if record["gen_ai.operation.name"] == "invoke_agent":
        print(f"step {record['step']:2d}  model call   status={record['status']}  "
              f"in={record['gen_ai.usage.input_tokens']}  out={record['gen_ai.usage.output_tokens']}")
    else:
        print(f"         tool call    {record['gen_ai.tool.name']:10s} {record['permission']:6s} "
              f"{record['duration_ms']:5d} ms  {record['result_bytes']:5d} bytes")

print("\n=== BUDGET ===")
print(f"steps={budget.steps}/{budget.max_steps}  tokens={budget.tokens}  "
      f"cost_usd={budget.cost_usd:.4f}  seconds={time.monotonic() - budget.started:.1f}")

print("\n=== progress.md ===")
print(PROGRESS.read_text())

print("=== pytest -q ===")
print(subprocess.run(["pytest", "-q"], capture_output=True, text=True).stdout.strip())

## Summary

In this notebook, we implemented:

1. **Minimal Harness**: A strict function-tool definition, a dispatcher that returns a bounded 100-line window, and the loop that resends history on every call
2. **Outcome Classification**: A `classify()` function that separates a finished task from truncation, refusal, and failure
3. **Budgets**: Step, token, cost, and wall-clock limits enforced by the harness, never delegated to the model
4. **Loop Detection**: A sliding-window hash of tool calls that nudges the model instead of hiding the tool
5. **Reinjection**: A verifier that rejects a claim of completion and feeds the failure back into the history
6. **Compaction**: Clearing old tool results while keeping the transcript valid and the results recoverable
7. **Externalized State**: A recited progress file for the model and an exact checkpoint for the harness
8. **Model-Facing Errors**: An `edit_file` tool that validates before writing and explains what to do next
9. **Permissions and Tracing**: An allow, ask, or deny gate matched to blast radius, and OpenTelemetry-style trace records
10. **The Hardened Loop**: All of the above assembled into one loop that fixes a real bug and stops only when the tests pass

### Key Takeaways:

- An agent is a model plus a harness; the harness owns everything the model cannot do for itself
- A response with no tool calls is a claim of completion, and only a verifier the harness trusts can accept it
- Budgets, loop detection, and permission checks belong in code, because the model cannot count steps, see the clock, or grant itself authority
- What survives compaction is a policy: instruction files, progress notes, checkpoints, and recent turns
- Tools should bound what they return and say what to do when they fail
- Traces with standard attribute names are the evidence that tells a harness failure from a model failure

### Next Steps:

- Replace the string-matching deny list with a parsed command policy and a real sandbox
- Add a summary turn when clearing tool results is no longer enough
- Ship the trace records to an OpenTelemetry backend and read them as a tree of spans
- Move on to Chapter 4, where the Model Context Protocol standardizes the tools a harness owns outright